# 9 Expert Global Fan-out Prompt Step-by-Step

这份 notebook 按你之前代码的思路构建 optical prompt：

```text
fanout_phase = angle(sum_j grating_j)
A_region = 3x3 region-wise amplitude weights
Ltot = A_region * exp(i * fanout_phase) * global_lens
Uout = fftshift(ifft2(fft2(flip(input)) * fft2(Ltot)))
```

它的物理含义更接近 **4f / convolution optical layer / global fan-out**，不是把输入手动复制 9 份，也不是把输入切成 9 个 patch。

重点区别：

- 之前那份 ASM prompt notebook：`input -> angular spectrum -> prompt plane -> pointwise prompt modulation -> angular spectrum -> experts`。
- 这份 notebook：`input -> convolution with global fan-out optical kernel -> 9 experts -> angular spectrum identity expert propagation`。

也就是说，本 notebook 的 expert 入口由你之前代码的卷积公式产生；expert 后续传播仍然使用当前项目里的角谱法。

结合 Zotero 中这篇论文的正文和附录来看，第一卷积层的 9 个输出更接近：

```text
P_total(x,y) = sum_{m,n} V_{m,n}(x,y) R_{m,n}(x,y) G_{m,n}(x,y)
phase_mask = angle(P_total) + quadratic_lens_phase
```

也就是 **9 个 diffraction directions / kernels 在 pupil function 中复数叠加，然后取总相位**。论文里确实有“zone / region”的说法，但更明确的分区发生在第二卷积层：第一层已经产生 3x3 feature maps，第二层再按这些 3x3 输出位置分成 3x3 zones。

In [ ]:
%matplotlib inline

from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.patches import Rectangle

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd() / 'opticalmoe']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'src' / 'opticalmoe').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot find opticalmoe/src. Please run this notebook inside the repository.')

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from opticalmoe.optics.angular_spectrum import AngularSpectrumPropagator

torch.set_grad_enabled(False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PROJECT_ROOT =', PROJECT_ROOT)
print('device =', device)

## 1. Geometry and Physical Parameters

默认使用：

- 输入：中心 `200 x 200`
- experts：`3 x 3`，每个 `200 x 200`
- expert bank：中心 `600 x 600`
- padding：外圈 `300 px`
- 总画布：`1200 x 1200`
- 焦距：默认 `10 cm`
- grating 目标复制间隔：`200 px = 1.6 mm`

注意：如果 `FOCAL_LENGTH_CM=10` 且成像距离用 `d=2f=20 cm`，这是薄透镜 1:1 成像的标准配置。

这里的 grating 频率按 `shift = wavelength * focal_length * f1` 进行设置。原因是本 notebook 采用 `fftshift(ifft2(fft2(flip(input))*fft2(Ltot)))` 的单透镜卷积实现，实测复制像 pitch 服从 focal-length 版本；如果直接用 `shift = wavelength * d * f1`，9 个复制像会明显挤向中心。

In [ ]:
WAVELENGTH_M = 532e-9
PIXEL_SIZE_M = 8e-6

INPUT_SIZE = 200
EXPERT_SIZE = 200
BANK_SIZE = 3 * EXPERT_SIZE
PADDING = 300
CANVAS_SIZE = BANK_SIZE + 2 * PADDING
CANVAS_SHAPE = (CANVAS_SIZE, CANVAS_SIZE)
CANVAS_CENTER = (CANVAS_SIZE // 2, CANVAS_SIZE // 2)

CENTER_COORDS = [PADDING + EXPERT_SIZE // 2, PADDING + 3 * EXPERT_SIZE // 2, PADDING + 5 * EXPERT_SIZE // 2]
EXPERT_IDS = ['E00','E01','E02','E10','E11','E12','E20','E21','E22']

FOCAL_LENGTH_CM = 10.0
CONVOLUTION_DISTANCE_CM = 2.0 * FOCAL_LENGTH_CM
INTER_LAYER_CM = 5.0
NUM_IDENTITY_EXPERT_LAYERS = 5

# 希望 9 个复制像的中心间距正好等于 expert pitch。
# 在本 notebook 使用的 FFT 卷积实现里，复制像位移实测更接近
#     shift = wavelength * focal_length * grating_frequency
# 而不是直接用 prompt-to-output 的 d=2f。因此 f1 用 focal_length 计算。
# 你之前参考代码里 delta_target=2 mm 但注释写 1 mm，也等价于做了这个 2 倍补偿。
COPY_PITCH_PX = EXPERT_SIZE
FANOUT_SIGN_X = 1.0
FANOUT_SIGN_Y = 1.0

# 如果 True，prompt/lens 只在中心 600x600 真实光学区域生效，外圈是 padding。
USE_CENTER_600_APERTURE = True

# fanout 相位来自 sum_j G_j 的相位；振幅始终是 3x3 区域常数权重。
PROMPT_MODE = 'region_phase_only'

AMPLITUDE_CASE = 'uniform'  # uniform / center_only / onehot_E00 / onehot_E22 / diagonal

print('canvas:', CANVAS_SHAPE)
print('expert centers:', CENTER_COORDS)
print('focal length cm:', FOCAL_LENGTH_CM)
print('convolution distance d=2f cm:', CONVOLUTION_DISTANCE_CM)
print('target copy pitch px:', COPY_PITCH_PX)

In [ ]:
def cm_to_m(value):
    return float(value) * 1e-2

def centered_slice(center, size):
    half = size // 2
    return slice(center - half, center + half)

def make_apertures(size):
    apertures = []
    for y in CENTER_COORDS:
        for x in CENTER_COORDS:
            apertures.append((centered_slice(y, size), centered_slice(x, size)))
    return apertures

EXPERT_APERTURES = make_apertures(EXPERT_SIZE)
PROMPT_APERTURES = make_apertures(EXPERT_SIZE)

def make_union_mask(apertures):
    mask = torch.zeros(CANVAS_SHAPE, dtype=torch.float32, device=device)
    for ys, xs in apertures:
        mask[ys, xs] = 1.0
    return mask

EXPERT_UNION_MASK = make_union_mask(EXPERT_APERTURES)
CENTER_600_MASK = make_union_mask(PROMPT_APERTURES)

def plot_layout():
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_facecolor('black')
    ys = centered_slice(CANVAS_CENTER[0], INPUT_SIZE)
    xs = centered_slice(CANVAS_CENTER[1], INPUT_SIZE)
    ax.add_patch(Rectangle((xs.start, ys.start), INPUT_SIZE, INPUT_SIZE, fill=False, edgecolor='orange', linewidth=2))
    for idx, (ys, xs) in enumerate(EXPERT_APERTURES):
        ax.add_patch(Rectangle((xs.start, ys.start), EXPERT_SIZE, EXPERT_SIZE, fill=False, edgecolor='lime', linewidth=1.2))
        ax.text(xs.start + 8, ys.start + 20, EXPERT_IDS[idx], color='lime', fontsize=9)
    ax.axhline(CANVAS_CENTER[0], color='white', alpha=0.25)
    ax.axvline(CANVAS_CENTER[1], color='white', alpha=0.25)
    ax.set_xlim(0, CANVAS_SIZE)
    ax.set_ylim(CANVAS_SIZE, 0)
    ax.set_title('1200x1200 canvas: center 600x600 optical area + 300px padding')
    plt.show()

plot_layout()

## 2. Build a 200x200 Digit Input

优先读取本地 MNIST；如果本地没有，就生成一个 digit-like 图案。

In [ ]:
def synthetic_digit(size=200):
    img = torch.zeros(size, size, dtype=torch.float32)
    t = max(8, size // 13)
    m = max(12, size // 7)
    img[m:m+t, m:size-m] = 1.0
    img[m:size//2, m:m+t] = 1.0
    img[size//2-t//2:size//2+t//2, m:size-m] = 1.0
    img[size//2:size-m, size-m-t:size-m] = 1.0
    img[size-m-t:size-m, m:size-m] = 1.0
    img[int(size*0.68):int(size*0.76), int(size*0.35):int(size*0.52)] = 0.55
    return img

def load_mnist_or_synthetic(index=0, size=200):
    try:
        from torchvision.datasets import MNIST
        dataset = MNIST(root=str(PROJECT_ROOT / 'data'), train=False, download=False)
        img = dataset.data[index].float() / 255.0
        img = F.interpolate(img[None, None], size=(size, size), mode='bilinear', align_corners=False)[0, 0]
        print('Loaded MNIST sample, label =', int(dataset.targets[index]))
        return img
    except Exception as exc:
        print('MNIST not available, using synthetic digit. Reason:', repr(exc))
        return synthetic_digit(size)

input_img = load_mnist_or_synthetic(index=0, size=INPUT_SIZE).to(device)
input_field = torch.zeros((1, CANVAS_SIZE, CANVAS_SIZE), dtype=torch.complex64, device=device)
ys = centered_slice(CANVAS_CENTER[0], INPUT_SIZE)
xs = centered_slice(CANVAS_CENTER[1], INPUT_SIZE)
input_field[0, ys, xs] = input_img.to(torch.complex64)

plt.figure(figsize=(4, 4))
plt.imshow(input_img.detach().cpu(), cmap='gray')
plt.title('200x200 input amplitude')
plt.axis('off')
plt.show()

## 3. Visualization and Expert Propagation Helpers

In [ ]:
def make_propagator(distance_cm):
    return AngularSpectrumPropagator(
        wavelength_m=WAVELENGTH_M,
        pixel_size_m=PIXEL_SIZE_M,
        grid_size=CANVAS_SHAPE,
        distance_m=cm_to_m(distance_cm),
    ).to(device)

prop_inter_layer = make_propagator(INTER_LAYER_CM)
print('Expert identity propagation uses:', type(prop_inter_layer).__name__)

def intensity(field):
    if torch.is_complex(field):
        return torch.abs(field).square()
    return field.float()

def log_intensity(field):
    arr = intensity(field)
    if arr.ndim == 3:
        arr = arr[0]
    arr = arr.detach().cpu().numpy()
    return np.log10(arr / (arr.max() + 1e-12) + 1e-8)

def wrapped_phase(phase_or_field):
    if torch.is_complex(phase_or_field):
        phase = torch.angle(phase_or_field)
    else:
        phase = phase_or_field
    if phase.ndim == 3:
        phase = phase[0]
    return torch.remainder(phase, 2 * math.pi).detach().cpu().numpy()

def overlay_experts(ax, color='lime'):
    for idx, (ys, xs) in enumerate(EXPERT_APERTURES):
        ax.add_patch(Rectangle((xs.start, ys.start), EXPERT_SIZE, EXPERT_SIZE, fill=False, edgecolor=color, linewidth=0.8))
        ax.text(xs.start + 5, ys.start + 18, EXPERT_IDS[idx], color=color, fontsize=7)

def show_field(field, title='', figsize=(6, 6)):
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(log_intensity(field), cmap='inferno')
    overlay_experts(ax)
    ax.set_title(title)
    ax.set_xlim(0, CANVAS_SIZE)
    ax.set_ylim(CANVAS_SIZE, 0)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label='log intensity')
    plt.show()

def show_phase(field_or_phase, title=''):
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(wrapped_phase(field_or_phase), cmap='twilight', vmin=0, vmax=2*math.pi)
    overlay_experts(ax)
    ax.set_title(title)
    ax.set_xlim(0, CANVAS_SIZE)
    ax.set_ylim(CANVAS_SIZE, 0)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    plt.show()

def expert_energy(field):
    I = intensity(field)[0]
    values = []
    for ys, xs in EXPERT_APERTURES:
        values.append(float(I[ys, xs].sum().detach().cpu()))
    values = np.asarray(values, dtype=np.float64)
    return values / (values.sum() + 1e-12)

def show_energy_heatmap(values, title='expert energy ratio'):
    arr = np.asarray(values).reshape(3, 3)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    im = ax.imshow(arr, cmap='magma')
    for r in range(3):
        for c in range(3):
            ax.text(c, r, f'{arr[r,c]:.3f}', ha='center', va='center', color='white')
    ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    plt.show()

def expert_centroid_table(field):
    """Return per-expert energy ratio, centroid, and center error in pixels."""
    I = intensity(field)[0]
    rows = []
    energies = []
    for ys, xs in EXPERT_APERTURES:
        energies.append(float(I[ys, xs].sum().detach().cpu()))
    energy_sum = sum(energies) + 1e-12
    for idx, (ys, xs) in enumerate(EXPERT_APERTURES):
        crop = I[ys, xs]
        e = crop.sum() + 1e-12
        yy_local = torch.arange(ys.start, ys.stop, device=device).float()
        xx_local = torch.arange(xs.start, xs.stop, device=device).float()
        YY_local, XX_local = torch.meshgrid(yy_local, xx_local, indexing='ij')
        cy = float((crop * YY_local).sum().detach().cpu() / e.detach().cpu())
        cx = float((crop * XX_local).sum().detach().cpu() / e.detach().cpu())
        target_y = CENTER_COORDS[idx // 3]
        target_x = CENTER_COORDS[idx % 3]
        err = math.sqrt((cy - target_y) ** 2 + (cx - target_x) ** 2)
        rows.append({
            'expert': EXPERT_IDS[idx],
            'energy_ratio': energies[idx] / energy_sum,
            'centroid_y': cy,
            'centroid_x': cx,
            'target_y': target_y,
            'target_x': target_x,
            'center_error_px': err,
        })
    return rows

def print_expert_centroid_table(field, title='centroid table'):
    rows = expert_centroid_table(field)
    print('\n' + title)
    print('expert  energy   centroid_y  centroid_x  target_y  target_x  error_px')
    for row in rows:
        print(f"{row['expert']:>4s}  {row['energy_ratio']:.4f}   {row['centroid_y']:9.1f}  {row['centroid_x']:9.1f}  {row['target_y']:8.1f}  {row['target_x']:8.1f}  {row['center_error_px']:8.1f}")
    errors = [row['center_error_px'] for row in rows]
    active_errors = [row['center_error_px'] for row in rows if row['energy_ratio'] > 0.01]
    print(f'all-region mean error: {np.mean(errors):.2f} px, max error: {np.max(errors):.2f} px')
    if active_errors:
        print(f'active-region (>1% energy) mean error: {np.mean(active_errors):.2f} px, max error: {np.max(active_errors):.2f} px')
    else:
        print('active-region (>1% energy) mean error: n/a')

## 4. Build the Global Fan-out Prompt

这里先用 `sum_j G_j` 得到 global fan-out 的相位，再用一个单独的 `3x3 region amplitude map` 控制空间区域振幅。这样振幅图始终是分区常数权重，不会出现光栅条纹；外圈 padding 始终为 0。

重要区别：`region amplitude` 控制的是 prompt 面上 3x3 空间区域的透过率；它不是严格意义上的 `order amplitude`，因此不能保证只打开某个 diffraction order。真正按论文式 `sum_j a_j G_j` 控制 9 个复制方向时，`a_j` 应该乘在每个全局 grating/order 上，但那样 `abs(sum_j a_j G_j)` 会出现干涉条纹，不再是分区常数振幅。

In [ ]:
yy = (torch.arange(CANVAS_SIZE, device=device).float() - CANVAS_CENTER[0]) * PIXEL_SIZE_M
xx = (torch.arange(CANVAS_SIZE, device=device).float() - CANVAS_CENTER[1]) * PIXEL_SIZE_M
Y, X = torch.meshgrid(yy, xx, indexing='ij')

AMPLITUDE_CASES = {
    'uniform': [1,1,1,1,1,1,1,1,1],
    'center_only': [0,0,0,0,1,0,0,0,0],
    'onehot_E00': [1,0,0,0,0,0,0,0,0],
    'onehot_E22': [0,0,0,0,0,0,0,0,1],
    'diagonal': [1,0,0,0,1,0,0,0,1],
}

def region_amplitude_map(amplitudes):
    amap = torch.zeros(CANVAS_SHAPE, dtype=torch.float32, device=device)
    for amp, (ys, xs) in zip(amplitudes, PROMPT_APERTURES):
        amap[ys, xs] = float(amp)
    return amap

def build_global_fanout_prompt(amplitude_case='uniform', prompt_mode='region_phase_only'):
    amplitudes = AMPLITUDE_CASES[amplitude_case]
    k = 2 * math.pi / WAVELENGTH_M
    f = cm_to_m(FOCAL_LENGTH_CM)
    d = cm_to_m(CONVOLUTION_DISTANCE_CM)

    target_copy_pitch_m = COPY_PITCH_PX * PIXEL_SIZE_M
    # 这里用 focal length 计算 f1，是为了让 FFT 卷积输出的实测 pitch 对齐 expert pitch。
    # 如果改回 WAVELENGTH_M * d，当前数值会让 9 个复制像挤向中心，约只产生一半位移。
    f1 = target_copy_pitch_m / (WAVELENGTH_M * f)

    orders = [(-1,-1),(0,-1),(1,-1),
              (-1, 0),(0, 0),(1, 0),
              (-1, 1),(0, 1),(1, 1)]

    # P_sum 只用于生成 fan-out phase。这里不把 amplitude 混进 P_sum，
    # 因为当前希望振幅始终是 3x3 region-wise 常数权重。
    P_sum = torch.zeros(CANVAS_SHAPE, dtype=torch.complex64, device=device)
    for _amp, (m, n) in zip(amplitudes, orders):
        G = torch.exp(1j * 2 * math.pi * (FANOUT_SIGN_X * m * f1 * X + FANOUT_SIGN_Y * n * f1 * Y)).to(torch.complex64)
        P_sum = P_sum + G

    if prompt_mode != 'region_phase_only':
        raise ValueError(prompt_mode)
    fanout_phase = torch.angle(P_sum) * CENTER_600_MASK
    region_amp = region_amplitude_map(amplitudes)
    P_used = region_amp.to(torch.complex64) * torch.exp(1j * fanout_phase).to(torch.complex64)

    lens_phase = (-k / (2 * f) * (X**2 + Y**2)) * CENTER_600_MASK
    lens = torch.exp(1j * lens_phase).to(torch.complex64) * CENTER_600_MASK.to(torch.complex64)
    Ltot = P_used * lens

    return {
        'P_sum': P_sum * CENTER_600_MASK.to(torch.complex64),
        'P_used': P_used,
        'fanout_phase': fanout_phase,
        'region_amplitude_map': region_amp,
        'lens_phase': lens_phase,
        'lens': lens,
        'Ltot': Ltot,
        'amplitudes': amplitudes,
        'f1': f1,
        'target_copy_pitch_m': target_copy_pitch_m,
        'paper_d_formula_pitch_m': WAVELENGTH_M * d * f1,
        'focal_formula_pitch_m': WAVELENGTH_M * f * f1,
    }

prompt = build_global_fanout_prompt(AMPLITUDE_CASE, PROMPT_MODE)
print('prompt mode:', PROMPT_MODE)
print('amplitude case:', AMPLITUDE_CASE)
print('target copy spacing mm:', prompt['target_copy_pitch_m'] * 1e3)
print('focal-formula spacing mm:', prompt['focal_formula_pitch_m'] * 1e3)
print('d=2f formula would report mm:', prompt['paper_d_formula_pitch_m'] * 1e3)
print('grating spatial frequency f1:', prompt['f1'])
show_phase(prompt['fanout_phase'], 'fan-out phase from sum_j grating_j, center 600x600 only')
show_field(prompt['region_amplitude_map'].unsqueeze(0), '3x3 region amplitude map, padding is zero')
show_phase(prompt['lens_phase'], 'global lens phase, center 600x600 only')
show_phase(prompt['Ltot'], 'total prompt phase: A_region * exp(i*fanout_phase) * lens')

## 5. Apply the Convolution Prompt and Continue Expert Propagation

In [ ]:
def run_global_fanout_pipeline(amplitude_case='uniform', prompt_mode='region_phase_only'):
    prompt = build_global_fanout_prompt(amplitude_case, prompt_mode)
    U0 = input_field[0]
    U0_flip = torch.flip(U0, dims=[0, 1])
    Uf = torch.fft.fft2(U0_flip)
    Pf = torch.fft.fft2(prompt['Ltot'])
    expert_entrance = torch.fft.fftshift(torch.fft.ifft2(Uf * Pf)).unsqueeze(0)

    after_aperture = expert_entrance * EXPERT_UNION_MASK.unsqueeze(0).to(torch.complex64)
    layers = [after_aperture]
    field = after_aperture
    for _ in range(NUM_IDENTITY_EXPERT_LAYERS):
        field = prop_inter_layer(field)
        field = field * EXPERT_UNION_MASK.unsqueeze(0).to(torch.complex64)
        layers.append(field)

    return {
        'prompt': prompt,
        'expert_entrance': expert_entrance,
        'after_aperture': after_aperture,
        'layers': layers,
        'detector': layers[-1],
        'expert_energy': expert_energy(expert_entrance),
        'detector_energy': expert_energy(layers[-1]),
    }

result = run_global_fanout_pipeline(AMPLITUDE_CASE, PROMPT_MODE)
show_field(input_field, 'input field')
show_field(result['expert_entrance'], 'expert entrance from global fan-out convolution')
show_energy_heatmap(result['expert_energy'], 'expert entrance energy')
print_expert_centroid_table(result['expert_entrance'], 'expert entrance copy alignment')
show_field(result['layers'][1], 'after identity expert propagation layer 1')
show_field(result['layers'][min(3, len(result['layers'])-1)], 'after identity expert propagation layer 3')
show_field(result['detector'], 'detector plane')
show_energy_heatmap(result['detector_energy'], 'detector energy')

## 6. Amplitude Control Test

In [ ]:
for case in ['uniform', 'center_only', 'onehot_E00', 'onehot_E22', 'diagonal']:
    res = run_global_fanout_pipeline(case, PROMPT_MODE)
    print('\n===', case, '===')
    print('entrance energy:', dict(zip(EXPERT_IDS, np.round(res['expert_energy'], 3))))
    print('detector energy:', dict(zip(EXPERT_IDS, np.round(res['detector_energy'], 3))))
    print_expert_centroid_table(res['expert_entrance'], case + ' entrance alignment')
    show_energy_heatmap(res['expert_energy'], case + ' expert entrance energy')
    show_field(res['expert_entrance'], case + ' expert entrance')

## 7. Interpretation

如果这里能看到 9 份清晰复制，而原 ASM 分区 prompt notebook 看不到，核心原因不是焦距或角谱传播本身，而是 prompt 结构不同：

- 原 ASM 分区 prompt：每个局部 region 只能调制到达该 region 的光。中心 200x200 输入传播到 prompt 平面后，非中心 region 通常没有足够光。
- 本 notebook：fan-out phase 来自 `angle(sum_j G_j)`，每个 grating order 都参与完整输入的卷积，因此可以生成完整输入的多个平移副本。

这里的振幅不是 `abs(sum_j G_j)`，而是单独的 3x3 region-wise amplitude map；padding 区域振幅为 0。需要注意：这种 region amplitude 控制的是空间区域透过率，不完全等价于独立控制每个 diffraction order 的效率。

如果你看到 9 个像偏向中心，优先检查 `target copy pitch px` 和 `focal-formula spacing` 是否等于 `200 px = 1.6 mm`。当前 notebook 已经按 focal-length 公式修正，质心表中的 `center_error_px` 应该明显小于之前。